# Distribution-Aware Open-World Object Detection
## Contribution A - Distribution-Aware Active Annotation

Closed-world object detection assumes every object category at test time was known during training. Open-world object detection also asks the detector to identify unknown objects, then improve as those objects become annotated in later tasks.

Contribution A targets the active-annotation step under a long-tail unknown-class distribution. A rare unknown class is valuable, but rarity alone can over-select isolated outliers. The acquisition score used here is:

$$
\text{score}
=
\alpha \cdot \text{uncertainty}
+ \beta \cdot \text{novelty}
+ \gamma \cdot \text{rarity} \cdot \text{coherence}^{p}
$$

The key semantic point is that local coherence gates only the rarity bonus. Low coherence suppresses `rarity * coherence**p`; it does not suppress uncertainty or novelty.

This notebook validates the detector-independent Contribution A pipeline on deterministic synthetic data, checks dataset-state and grouped unknown-recall utilities, prepares the PROB bridge, and optionally runs a one-image PROB proposal export when the required Google Drive data and checkpoint are present.

It does not claim full PROB train/evaluate integration. Full training, official known mAP, WI, A-OSE, and standard U-Recall remain delegated to the official PROB code and are marked pending here.

## 1. Runtime And Reproducibility

In [ ]:
import os
import platform
import random
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_SEED = 7
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

try:
    import torch
except ModuleNotFoundError:
    torch = None

if torch is not None:
    torch.manual_seed(RANDOM_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(RANDOM_SEED)

RUNTIME = {
    "Python": sys.version.split()[0],
    "Platform": platform.platform(),
    "PyTorch": getattr(torch, "__version__", "not installed"),
    "CUDA version": getattr(getattr(torch, "version", None), "cuda", None)
    if torch is not None
    else "not installed",
    "CUDA available": bool(torch is not None and torch.cuda.is_available()),
    "GPU": torch.cuda.get_device_name(0)
    if torch is not None and torch.cuda.is_available()
    else "not available",
    "Seed": RANDOM_SEED,
}
display(pd.DataFrame([RUNTIME]).T.rename(columns={0: "value"}))
print("Seeds are fixed for Python, NumPy, and PyTorch when installed.")
print("CUDA operations are requested deterministically where practical, but perfect CUDA determinism is not guaranteed.")

STATUS = {
    "GPU": "OK" if RUNTIME["CUDA available"] else "SKIPPED",
    "DAOWOD clone": "SKIPPED",
    "DAOWOD installation": "SKIPPED",
    "Ruff": "SKIPPED",
    "pytest": "SKIPPED",
    "compileall": "SKIPPED",
    "synthetic scoring": "SKIPPED",
    "ablation demo": "SKIPPED",
    "dataset-state demo": "SKIPPED",
    "grouped metrics demo": "SKIPPED",
    "PROB clone": "SKIPPED",
    "PROB patch": "SKIPPED",
    "bridge check": "SKIPPED",
    "PROB attention backend": "SKIPPED",
    "Drive": "SKIPPED",
    "dataset": "MISSING",
    "checkpoint": "MISSING",
    "real proposal export": "SKIPPED",
    "real feature scoring": "SKIPPED",
    "full train/evaluate loop": "NOT IMPLEMENTED",
}

def run_command(command, *, cwd=None, timeout=600, check=True, preview_lines=40):
    command = [str(part) for part in command]
    print("$", " ".join(command))
    result = subprocess.run(
        command,
        cwd=cwd,
        text=True,
        capture_output=True,
        timeout=timeout,
    )
    combined = "\n".join(part for part in (result.stdout, result.stderr) if part)
    if combined:
        lines = combined.splitlines()
        shown = lines[-preview_lines:]
        print("\n".join(shown))
        if len(lines) > preview_lines:
            print(f"... trimmed {len(lines) - preview_lines} earlier lines")
    print("return code:", result.returncode)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed: {' '.join(command)}")
    return result

## 2. Configuration

In [ ]:
try:
    import google.colab  # type: ignore[import-not-found]

    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

DAOWOD_REPOSITORY_URL = "https://github.com/gubiczam/distribution-aware-owod.git"
DAOWOD_BRANCH = "main"

PROB_REPOSITORY_URL = "https://github.com/orrzohar/PROB.git"
PROB_TESTED_COMMIT = "b1d7fe68f5d55ffe8bc996ab15d7a260fee1e30a"
PROB_FORK_URL = "https://github.com/gubiczam/PROB.git"

CONTENT_ROOT = Path("/content") if IN_COLAB else Path.cwd()
DRIVE_ROOT = Path("/content/drive/MyDrive/DAOWOD")
DRIVE_DATA_ROOT = DRIVE_ROOT / "data" / "OWOD"
DRIVE_CHECKPOINT = DRIVE_ROOT / "checkpoints" / "MOWODB" / "t1.pth"

TASK_SETTINGS = {
    "dataset": "TOWOD",
    "test_split": "owod_all_task_test",
    "random_seed": RANDOM_SEED,
    "budget": 2,
    "top_k": 2,
    "max_proposals_per_image": 20,
    "minimum_unknown_score": 0.0,
}

CONFIGURATION = {
    "works_immediately": {
        "DAOWOD repository": DAOWOD_REPOSITORY_URL,
        "DAOWOD branch": DAOWOD_BRANCH,
        "PROB repository": PROB_REPOSITORY_URL,
        "PROB commit": PROB_TESTED_COMMIT,
        "synthetic demos": True,
        "local tests": True,
    },
    "requires_user_assets": {
        "Drive root": str(DRIVE_ROOT),
        "dataset root": str(DRIVE_DATA_ROOT),
        "checkpoint": str(DRIVE_CHECKPOINT),
    },
    "task_settings": TASK_SETTINGS,
}
display(pd.DataFrame(CONFIGURATION["works_immediately"].items(), columns=["setting", "value"]))
display(pd.DataFrame(CONFIGURATION["requires_user_assets"].items(), columns=["setting", "value"]))
display(pd.DataFrame(TASK_SETTINGS.items(), columns=["setting", "value"]))

## 3. Safe Repository Setup

In [ ]:
def find_daowod_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "daowod").exists():
            return candidate
    raise FileNotFoundError("Could not locate the DAOWOD repository root.")

if IN_COLAB:
    os.chdir("/content")
    for clone_name in ("distribution-aware-owod", "PROB"):
        clone_path = Path("/content") / clone_name
        if clone_path.exists():
            if Path.cwd().resolve() == clone_path.resolve() or clone_path.resolve() in Path.cwd().resolve().parents:
                os.chdir("/content")
            import shutil

            shutil.rmtree(clone_path)

    run_command(
        [
            "git",
            "clone",
            "--branch",
            DAOWOD_BRANCH,
            "--single-branch",
            DAOWOD_REPOSITORY_URL,
            "distribution-aware-owod",
        ],
        cwd="/content",
        timeout=300,
    )
    run_command(["git", "clone", PROB_REPOSITORY_URL, "PROB"], cwd="/content", timeout=300)
    run_command(["git", "checkout", PROB_TESTED_COMMIT], cwd="/content/PROB", timeout=120)
    DAOWOD_PATH = Path("/content/distribution-aware-owod")
    PROB_PATH = Path("/content/PROB")
else:
    DAOWOD_PATH = find_daowod_root(Path.cwd().resolve())
    PROB_PATH = DAOWOD_PATH.parent / "PROB"

os.chdir(DAOWOD_PATH)

if not DAOWOD_PATH.exists():
    raise FileNotFoundError(f"DAOWOD path missing: {DAOWOD_PATH}")
STATUS["DAOWOD clone"] = "OK"

if PROB_PATH.exists():
    STATUS["PROB clone"] = "OK"
    prob_commit = run_command(["git", "rev-parse", "--short", "HEAD"], cwd=PROB_PATH, check=False).stdout.strip()
else:
    STATUS["PROB clone"] = "MISSING"
    prob_commit = "missing"

daowod_commit = run_command(["git", "rev-parse", "--short", "HEAD"], cwd=DAOWOD_PATH).stdout.strip()
print("DAOWOD:", DAOWOD_PATH, daowod_commit)
print("PROB:", PROB_PATH, prob_commit)

## 4. Install And Test DAOWOD

In [ ]:
try:
    run_command([sys.executable, "-m", "pip", "install", "--editable", ".[dev]"], cwd=DAOWOD_PATH, timeout=600)
    STATUS["DAOWOD installation"] = "OK"

    run_command(["ruff", "check", "."], cwd=DAOWOD_PATH, timeout=300)
    run_command(["ruff", "format", "--check", "."], cwd=DAOWOD_PATH, timeout=300)
    STATUS["Ruff"] = "OK"

    run_command(["pytest", "-q"], cwd=DAOWOD_PATH, timeout=300)
    STATUS["pytest"] = "OK"

    run_command([sys.executable, "-m", "compileall", "-q", "src"], cwd=DAOWOD_PATH, timeout=300)
    STATUS["compileall"] = "OK"
except Exception:
    if STATUS["DAOWOD installation"] != "OK":
        STATUS["DAOWOD installation"] = "FAILED"
    elif STATUS["Ruff"] != "OK":
        STATUS["Ruff"] = "FAILED"
    elif STATUS["pytest"] != "OK":
        STATUS["pytest"] = "FAILED"
    else:
        STATUS["compileall"] = "FAILED"
    raise

In [ ]:
import importlib
import sys

DAOWOD_SRC = DAOWOD_PATH / "src"

if str(DAOWOD_SRC) not in sys.path:
    sys.path.insert(0, str(DAOWOD_SRC))

importlib.invalidate_caches()

import daowod

print("DAOWOD import: OK")
print("Loaded from:", daowod.__file__)


## 5. Load The Experiment Config

In [ ]:
from daowod import load_config

experiment_config = load_config(DAOWOD_PATH / "configs" / "experiment.yaml")
config_summary = {
    "acquisition weights": experiment_config.acquisition.weights,
    "uncertainty mode": experiment_config.acquisition.uncertainty_mode,
    "pseudo-label source": experiment_config.acquisition.pseudo_label_source,
    "top-k aggregation": experiment_config.acquisition.top_k,
    "budget per round": experiment_config.active_learning.budget_per_round,
    "rounds": experiment_config.active_learning.rounds,
    "seeds": experiment_config.active_learning.seeds,
    "long-tail settings": experiment_config.dataset.long_tail,
}
display(pd.DataFrame((str(k), str(v)) for k, v in config_summary.items()).rename(columns={0: "field", 1: "value"}))

## 6. Deterministic Synthetic Contribution A Demo

In [ ]:
import matplotlib.pyplot as plt

from daowod.acquisition import (
    AcquisitionWeights,
    aggregate_image_scores,
    score_proposals,
    select_images,
)

weights = AcquisitionWeights()
synthetic_image_ids = np.array(
    [
        "common_1",
        "common_1",
        "common_2",
        "common_2",
        "common_3",
        "common_3",
        "rare_1",
        "rare_1",
        "outlier_1",
    ],
    dtype=object,
)
synthetic_embeddings = np.array(
    [
        [1.00, 0.00],
        [0.98, 0.05],
        [0.96, -0.03],
        [1.03, 0.02],
        [0.97, 0.04],
        [1.01, -0.04],
        [-0.15, 1.00],
        [-0.12, 0.98],
        [-1.00, -1.00],
    ],
    dtype=float,
)
labelled_reference_embeddings = np.array([[1.00, 0.00], [0.95, 0.05]], dtype=float)
synthetic_confidence = np.array([0.87, 0.82, 0.79, 0.84, 0.81, 0.77, 0.52, 0.49, 0.50])
synthetic_predicted_labels = np.array([0, 0, 0, 0, 0, 0, 1, 1, 2])

synthetic_result = score_proposals(
    strategy="full",
    uncertainty_mode="ambiguity",
    pseudo_label_source="predicted",
    confidence=synthetic_confidence,
    posterior=None,
    embeddings=synthetic_embeddings,
    reference_embeddings=labelled_reference_embeddings,
    predicted_labels=synthetic_predicted_labels,
    cluster_count=3,
    neighbour_count=1,
    seed=RANDOM_SEED,
    weights=weights,
)
rarity_bonus = synthetic_result.rarity * synthetic_result.coherence**weights.coherence_power
synthetic_table = pd.DataFrame(
    {
        "image_id": synthetic_image_ids,
        "uncertainty": synthetic_result.uncertainty,
        "novelty": synthetic_result.novelty,
        "pseudo_label": synthetic_result.pseudo_labels,
        "rarity": synthetic_result.rarity,
        "coherence": synthetic_result.coherence,
        "rarity_bonus": rarity_bonus,
        "full_score": synthetic_result.scores,
    }
)
display(synthetic_table.round(4))

outlier = synthetic_table["image_id"] == "outlier_1"
rare = synthetic_table["image_id"] == "rare_1"
assert synthetic_table.loc[outlier, "coherence"].iloc[0] < synthetic_table.loc[rare, "coherence"].mean()
assert synthetic_table.loc[outlier, "rarity_bonus"].iloc[0] < synthetic_table.loc[outlier, "rarity"].iloc[0]
assert synthetic_table.loc[rare, "rarity_bonus"].mean() > synthetic_table.loc[outlier, "rarity_bonus"].iloc[0]
selected_images = select_images(
    synthetic_image_ids,
    synthetic_result.scores,
    budget=TASK_SETTINGS["budget"],
    top_k=TASK_SETTINGS["top_k"],
)
assert len(selected_images) == TASK_SETTINGS["budget"]
assert len(synthetic_image_ids) == len(synthetic_confidence) == len(synthetic_embeddings) == len(synthetic_result.scores)
STATUS["synthetic scoring"] = "OK"

image_scores = aggregate_image_scores(
    synthetic_image_ids,
    synthetic_result.scores,
    top_k=TASK_SETTINGS["top_k"],
)
print("Selected images:", selected_images)
display(pd.DataFrame(image_scores.items(), columns=["image_id", "image_score"]).sort_values("image_score", ascending=False).round(4))

In [ ]:
plt.figure()
plt.scatter(synthetic_embeddings[:, 0], synthetic_embeddings[:, 1])
for image_id, xy in zip(synthetic_image_ids, synthetic_embeddings, strict=True):
    plt.annotate(image_id, xy=xy)
plt.title("Synthetic Proposal Embeddings")
plt.xlabel("embedding dimension 1")
plt.ylabel("embedding dimension 2")
plt.show()

plt.figure()
synthetic_table[["uncertainty", "novelty", "rarity_bonus", "full_score"]].plot(kind="bar", ax=plt.gca())
plt.title("Proposal Score Components")
plt.xlabel("proposal index")
plt.ylabel("score")
plt.show()

plt.figure()
pd.Series(image_scores).sort_values().plot(kind="barh")
plt.title("Image-Level Ranking")
plt.xlabel("top-k mean proposal score")
plt.show()

## 7. Ablation Demo

In [ ]:
from daowod.acquisition import compute_proposal_scores

strategies = [
    "random",
    "uncertainty",
    "uncertainty_novelty",
    "rarity",
    "rarity_coherence",
    "ungated_full",
    "full",
]
rng = np.random.default_rng(RANDOM_SEED)
unique_images = np.array(list(dict.fromkeys(synthetic_image_ids.tolist())), dtype=object)
ablation_rows = []

for strategy in strategies:
    if strategy == "random":
        selected = rng.choice(unique_images, size=TASK_SETTINGS["budget"], replace=False).tolist()
    else:
        strategy_scores = compute_proposal_scores(
            strategy=strategy,
            uncertainty=synthetic_result.uncertainty,
            novelty=synthetic_result.novelty,
            rarity=synthetic_result.rarity,
            coherence=synthetic_result.coherence,
            weights=weights,
        )
        selected = select_images(
            synthetic_image_ids,
            strategy_scores,
            budget=TASK_SETTINGS["budget"],
            top_k=TASK_SETTINGS["top_k"],
        )
    selected_mask = np.isin(synthetic_image_ids, selected)
    ablation_rows.append(
        {
            "strategy": strategy,
            "selected images": ", ".join(str(value) for value in selected),
            "mean selected rarity": float(synthetic_result.rarity[selected_mask].mean()),
            "mean selected coherence": float(synthetic_result.coherence[selected_mask].mean()),
            "isolated-outlier selections": int("outlier_1" in selected),
        }
    )

ablation_table = pd.DataFrame(ablation_rows)
display(ablation_table.round(4))
STATUS["ablation demo"] = "OK"
print("This is a functional and semantic validation on toy data, not an experimental result.")

## 8. Dataset State And Controlled Long-Tail Demo

In [ ]:
import json
import tempfile
import xml.etree.ElementTree as ET

from daowod.dataset import (
    DatasetState,
    build_long_tail_pool,
    frequency_groups,
    read_image_ids,
    read_voc_classes,
    unknown_class_counts,
)

def write_voc_xml(path: Path, image_id: str, classes: list[str]) -> None:
    objects = []
    for class_name in classes:
        objects.append(
            f"<object><name>{class_name}</name><bndbox><xmin>1</xmin><ymin>1</ymin><xmax>10</xmax><ymax>10</ymax></bndbox></object>"
        )
    path.write_text(
        "<annotation>"
        f"<filename>{image_id}.jpg</filename>"
        "<size><width>100</width><height>100</height><depth>3</depth></size>"
        + "".join(objects)
        + "</annotation>",
        encoding="utf-8",
    )

with tempfile.TemporaryDirectory() as tmp:
    tmp_path = Path(tmp)
    annotations_dir = tmp_path / "Annotations"
    image_set_dir = tmp_path / "ImageSets" / "TOWOD"
    annotations_dir.mkdir(parents=True)
    image_set_dir.mkdir(parents=True)

    annotation_classes = {
        "img001": ["common_unknown", "known"],
        "img002": ["common_unknown"],
        "img003": ["common_unknown"],
        "img004": ["medium_unknown"],
        "img005": ["medium_unknown", "rare_unknown"],
    }
    for image_id, classes in annotation_classes.items():
        write_voc_xml(annotations_dir / f"{image_id}.xml", image_id, classes)

    image_set_path = image_set_dir / "toy_train.txt"
    image_set_path.write_text("\n".join(annotation_classes) + "\n", encoding="utf-8")

    image_ids = read_image_ids(image_set_path)
    counts = unknown_class_counts(
        image_ids,
        annotations_dir,
        ["common_unknown", "medium_unknown", "rare_unknown"],
    )
    class_groups = frequency_groups(counts, tail_max=1, head_min=3)
    manifest_path = tmp_path / "manifest.json"
    controlled_pool, pool_groups = build_long_tail_pool(
        image_ids,
        annotations_dir=annotations_dir,
        unknown_classes=["common_unknown", "medium_unknown", "rare_unknown"],
        tail_max=1,
        head_min=3,
        head_retention=0.5,
        medium_retention=0.5,
        tail_retention=1.0,
        seed=RANDOM_SEED,
        manifest_path=manifest_path,
    )
    state = DatasetState.initialise(controlled_pool, initial_images=1, seed=RANDOM_SEED)
    selected_for_reveal = state.pool_ids[:1]
    state.reveal(selected_for_reveal)

    original_object_counts = {image_id: len(classes) for image_id, classes in annotation_classes.items()}
    observed_object_counts = {
        image_id: len(read_voc_classes(image_id, annotations_dir)) for image_id in image_ids
    }
    assert observed_object_counts == original_object_counts
    assert set(controlled_pool).issubset(set(image_ids))
    assert selected_for_reveal[0] in state.labelled_ids
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))

display(pd.DataFrame(counts.items(), columns=["unknown_class", "count"]))
display(pd.DataFrame(class_groups.items(), columns=["unknown_class", "group"]))
display(pd.DataFrame({"controlled_pool": controlled_pool}))
print("Manifest keys:", sorted(manifest))
print("All annotations stayed image-level and intact; omitted images were not converted into partial background labels.")
STATUS["dataset-state demo"] = "OK"

## 9. Grouped Unknown-Recall Demo

In [ ]:
from daowod.metrics import Detection, GroundTruth, grouped_unknown_recall

toy_ground_truth = [
    GroundTruth("img001", "head_unknown", (0, 0, 10, 10)),
    GroundTruth("img002", "medium_unknown", (0, 0, 10, 10)),
    GroundTruth("img003", "tail_unknown", (0, 0, 10, 10)),
]
toy_detections = [
    Detection("img001", "unknown", 0.9, (20, 20, 30, 30)),
    Detection("img002", "unknown", 0.8, (0, 0, 10, 10)),
    Detection("img003", "unknown", 0.7, (1, 1, 11, 11)),
]
grouped_metrics = grouped_unknown_recall(
    toy_ground_truth,
    toy_detections,
    unknown_classes=["head_unknown", "medium_unknown", "tail_unknown"],
    class_groups={
        "head_unknown": "head",
        "medium_unknown": "medium",
        "tail_unknown": "tail",
    },
)
display(pd.DataFrame(grouped_metrics.items(), columns=["metric", "value"]).round(4))
STATUS["grouped metrics demo"] = "OK"
print("Known mAP, WI, A-OSE, and the official aggregate U-Recall remain delegated to PROB's evaluator.")

## 10. PROB Integration Setup

In [ ]:
BRIDGE_SOURCE = "\"\"\"Bridge from the official PROB checkout to the DAOWOD active-learning code.\n\nOnly ``predict`` exports proposal features today. ``train`` and ``evaluate`` are\ndeclared so experiment templates fail with a clear message until they are wired\nto the official PROB scripts for the selected OWOD protocol.\n\nThe local model patch must expose final decoder features with:\n\n    out[\"pred_features\"] = hs[-1]\n\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport os\nimport re\nimport sys\nfrom pathlib import Path\nfrom typing import Any\n\n\ndef read_image_ids(path: str | Path) -> list[str]:\n    \"\"\"Read one image identifier per line.\"\"\"\n\n    image_ids = [\n        line.strip().split()[0]\n        for line in Path(path).read_text(encoding=\"utf-8\").splitlines()\n        if line.strip()\n    ]\n    image_ids = list(dict.fromkeys(image_ids))\n    if not image_ids:\n        raise ValueError(f\"No image IDs found in {path}.\")\n    return image_ids\n\n\ndef check_feature_patch() -> None:\n    \"\"\"Check the decoder-feature export without importing PyTorch.\"\"\"\n\n    model_path = Path(__file__).resolve().parent / \"models\" / \"prob_deformable_detr.py\"\n    if not model_path.exists():\n        raise FileNotFoundError(f\"Missing PROB model file: {model_path}\")\n\n    source = model_path.read_text(encoding=\"utf-8\")\n    compile(source, str(model_path), \"exec\")\n    if \"'pred_features': hs[-1]\" not in source and '\"pred_features\": hs[-1]' not in source:\n        raise RuntimeError(\"pred_features export is missing from models/prob_deformable_detr.py\")\n\n    print(\"PROB decoder-feature patch: OK\")\n\n\ndef import_prob_runtime() -> dict[str, Any]:\n    \"\"\"Import heavy PROB dependencies only for commands that need inference.\"\"\"\n\n    try:\n        import numpy as np\n        import torch\n        from torch.utils.data import DataLoader, SequentialSampler\n\n        import util.misc as utils\n        from datasets.coco import make_coco_transforms\n        from datasets.torchvision_datasets.open_world import OWDetection\n        from main_open_world import get_args_parser\n        from models import build_model\n    except ModuleNotFoundError as error:\n        raise RuntimeError(\n            \"PROB prediction dependencies are unavailable. Run this command inside a \"\n            \"configured PROB environment.\"\n        ) from error\n\n    return {\n        \"np\": np,\n        \"torch\": torch,\n        \"DataLoader\": DataLoader,\n        \"SequentialSampler\": SequentialSampler,\n        \"utils\": utils,\n        \"make_coco_transforms\": make_coco_transforms,\n        \"OWDetection\": OWDetection,\n        \"get_args_parser\": get_args_parser,\n        \"build_model\": build_model,\n    }\n\n\ndef decode_original_image_id(target: dict[str, Any]) -> str:\n    \"\"\"Decode PROB's character-code representation of the original image ID.\"\"\"\n\n    values = target[\"org_image_id\"].detach().cpu().tolist()\n    return \"\".join(chr(int(value)) for value in values)\n\n\ndef create_prob_args(args: argparse.Namespace, get_args_parser: Any) -> argparse.Namespace:\n    \"\"\"Create the argument namespace expected by PROB's model and dataset.\"\"\"\n\n    prob_args = get_args_parser().parse_args([])\n    prob_args.device = args.device\n    prob_args.dataset = args.dataset\n    prob_args.data_root = str(Path(args.data_root).resolve())\n    prob_args.PREV_INTRODUCED_CLS = args.prev_introduced_classes\n    prob_args.CUR_INTRODUCED_CLS = args.current_introduced_classes\n    prob_args.num_classes = args.num_classes\n    prob_args.obj_temp = args.objectness_temperature\n    prob_args.batch_size = args.batch_size\n    prob_args.num_workers = args.num_workers\n    prob_args.model_type = \"prob\"\n    prob_args.seed = args.seed\n    return prob_args\n\n\ndef create_temporary_split(\n    *,\n    image_ids: list[str],\n    data_root: str | Path,\n    dataset_name: str,\n    output_path: str | Path,\n) -> tuple[str, Path]:\n    \"\"\"Create a temporary VOC-style split consumed by ``OWDetection``.\"\"\"\n\n    stem = re.sub(r\"[^A-Za-z0-9_.-]+\", \"_\", Path(output_path).stem)\n    split_name = f\"daowod_export_{stem}_{os.getpid()}_val\"\n    split_dir = Path(data_root) / \"ImageSets\" / dataset_name\n    split_dir.mkdir(parents=True, exist_ok=True)\n\n    split_path = split_dir / f\"{split_name}.txt\"\n    if split_path.exists():\n        raise FileExistsError(f\"Temporary split already exists: {split_path}\")\n    split_path.write_text(\"\\n\".join(image_ids) + \"\\n\", encoding=\"utf-8\")\n    return split_name, split_path\n\n\ndef remove_temporary_split(split_path: Path) -> None:\n    \"\"\"Remove only bridge-created split files.\"\"\"\n\n    if split_path.name.startswith(\"daowod_export_\") and split_path.exists():\n        split_path.unlink()\n\n\ndef load_model(\n    *,\n    prob_args: argparse.Namespace,\n    checkpoint_path: str | Path,\n    device: Any,\n    torch: Any,\n    build_model: Any,\n) -> Any:\n    \"\"\"Build PROB and load a training or evaluation checkpoint.\"\"\"\n\n    path = Path(checkpoint_path)\n    if not path.exists():\n        raise FileNotFoundError(f\"Missing checkpoint: {path}\")\n\n    model, _, _, _ = build_model(prob_args, mode=\"prob\")\n    checkpoint = torch.load(path, map_location=\"cpu\")\n    state_dict = checkpoint[\"model\"] if \"model\" in checkpoint else checkpoint\n    load_result = model.load_state_dict(state_dict, strict=False)\n\n    if load_result.missing_keys:\n        print(\"Missing checkpoint keys:\", load_result.missing_keys, file=sys.stderr)\n    if load_result.unexpected_keys:\n        print(\"Unexpected checkpoint keys:\", load_result.unexpected_keys, file=sys.stderr)\n\n    model.to(device)\n    model.eval()\n    return model\n\n\ndef valid_class_indices(\n    *,\n    previous_classes: int,\n    current_classes: int,\n    num_classes: int,\n    torch: Any,\n) -> Any:\n    \"\"\"Return known-class indices followed by PROB's unknown class index.\"\"\"\n\n    seen_class_count = previous_classes + current_classes\n    if seen_class_count < 1:\n        raise ValueError(\"At least one introduced class is required.\")\n    if seen_class_count >= num_classes:\n        raise ValueError(\"The introduced-class count must leave room for the unknown class.\")\n    return torch.tensor([*range(seen_class_count), num_classes - 1], dtype=torch.long)\n\n\ndef export_proposals(args: argparse.Namespace) -> None:\n    \"\"\"Run PROB and export query-level proposal features.\"\"\"\n\n    check_feature_patch()\n    if args.max_proposals_per_image < 1:\n        raise ValueError(\"--max-proposals-per-image must be positive.\")\n    if args.minimum_unknown_score < 0 or args.minimum_unknown_score > 1:\n        raise ValueError(\"--minimum-unknown-score must be in [0, 1].\")\n    if args.batch_size < 1:\n        raise ValueError(\"--batch-size must be positive.\")\n    if args.num_workers < 0:\n        raise ValueError(\"--num-workers must be non-negative.\")\n    if not Path(args.image_ids).exists():\n        raise FileNotFoundError(f\"Missing image ID file: {args.image_ids}\")\n    if not Path(args.checkpoint).exists():\n        raise FileNotFoundError(f\"Missing checkpoint: {args.checkpoint}\")\n    if not Path(args.data_root).exists():\n        raise FileNotFoundError(f\"Missing data root: {args.data_root}\")\n\n    runtime = import_prob_runtime()\n    np = runtime[\"np\"]\n    torch = runtime[\"torch\"]\n\n    output_path = Path(args.output).resolve()\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n\n    image_ids = read_image_ids(args.image_ids)\n    prob_args = create_prob_args(args, runtime[\"get_args_parser\"])\n    device = torch.device(args.device)\n\n    split_name, split_path = create_temporary_split(\n        image_ids=image_ids,\n        data_root=prob_args.data_root,\n        dataset_name=prob_args.dataset,\n        output_path=output_path,\n    )\n\n    try:\n        dataset = runtime[\"OWDetection\"](\n            prob_args,\n            prob_args.data_root,\n            image_set=split_name,\n            transforms=runtime[\"make_coco_transforms\"](split_name),\n            dataset=prob_args.dataset,\n        )\n        data_loader = runtime[\"DataLoader\"](\n            dataset,\n            batch_size=prob_args.batch_size,\n            sampler=runtime[\"SequentialSampler\"](dataset),\n            drop_last=False,\n            collate_fn=runtime[\"utils\"].collate_fn,\n            num_workers=prob_args.num_workers,\n            pin_memory=device.type == \"cuda\",\n        )\n        model = load_model(\n            prob_args=prob_args,\n            checkpoint_path=args.checkpoint,\n            device=device,\n            torch=torch,\n            build_model=runtime[\"build_model\"],\n        )\n        class_indices = valid_class_indices(\n            previous_classes=prob_args.PREV_INTRODUCED_CLS,\n            current_classes=prob_args.CUR_INTRODUCED_CLS,\n            num_classes=prob_args.num_classes,\n            torch=torch,\n        ).to(device)\n\n        all_image_ids: list[str] = []\n        all_confidence: list[Any] = []\n        all_embeddings: list[Any] = []\n        all_posterior: list[Any] = []\n        all_predicted_labels: list[Any] = []\n        all_boxes: list[Any] = []\n        all_objectness: list[Any] = []\n        temperature = prob_args.obj_temp / prob_args.hidden_dim\n\n        with torch.inference_mode():\n            for samples, targets in data_loader:\n                samples = samples.to(device)\n                outputs = model(samples)\n\n                required_outputs = {\"pred_logits\", \"pred_boxes\", \"pred_obj\", \"pred_features\"}\n                missing_outputs = required_outputs - set(outputs)\n                if missing_outputs:\n                    raise RuntimeError(\n                        f\"The patched PROB model is missing outputs: {sorted(missing_outputs)}\"\n                    )\n\n                logits = outputs[\"pred_logits\"]\n                boxes = outputs[\"pred_boxes\"]\n                objectness_distance = outputs[\"pred_obj\"]\n                features = outputs[\"pred_features\"]\n                if features.ndim != 3:\n                    raise RuntimeError(\n                        \"pred_features must have shape [batch, queries, hidden_dim].\"\n                    )\n\n                objectness_probability = torch.exp(-temperature * objectness_distance)\n                class_probability = logits.sigmoid().index_select(dim=2, index=class_indices)\n                combined_probability = objectness_probability.unsqueeze(-1) * class_probability\n                unknown_probability = combined_probability[:, :, -1]\n                keep_count = min(args.max_proposals_per_image, unknown_probability.shape[1])\n\n                top_values, top_queries = torch.topk(\n                    unknown_probability,\n                    k=keep_count,\n                    dim=1,\n                    largest=True,\n                    sorted=True,\n                )\n\n                for batch_index, target in enumerate(targets):\n                    keep = top_queries[batch_index]\n                    if args.minimum_unknown_score > 0:\n                        keep = keep[top_values[batch_index] >= args.minimum_unknown_score]\n                    if keep.numel() == 0:\n                        keep = top_queries[batch_index, :1]\n\n                    selected_combined = combined_probability[batch_index, keep]\n                    selected_posterior = selected_combined / selected_combined.sum(\n                        dim=1,\n                        keepdim=True,\n                    ).clamp_min(1e-12)\n                    compact_labels = selected_posterior.argmax(dim=1)\n                    selected_labels = class_indices[compact_labels]\n\n                    image_id = decode_original_image_id(target)\n                    proposal_count = keep.numel()\n                    all_image_ids.extend([image_id] * proposal_count)\n                    all_confidence.append(\n                        unknown_probability[batch_index, keep].detach().cpu().numpy()\n                    )\n                    all_embeddings.append(features[batch_index, keep].detach().cpu().numpy())\n                    all_posterior.append(selected_posterior.detach().cpu().numpy())\n                    all_predicted_labels.append(selected_labels.detach().cpu().numpy())\n                    all_boxes.append(boxes[batch_index, keep].detach().cpu().numpy())\n                    all_objectness.append(\n                        objectness_probability[batch_index, keep].detach().cpu().numpy()\n                    )\n\n        if not all_embeddings:\n            raise RuntimeError(\"PROB exported no proposals.\")\n\n        embeddings = np.concatenate(all_embeddings).astype(np.float64)\n        np.savez_compressed(\n            output_path,\n            image_ids=np.asarray(all_image_ids, dtype=object),\n            confidence=np.concatenate(all_confidence).astype(np.float64),\n            embeddings=embeddings,\n            posterior=np.concatenate(all_posterior).astype(np.float64),\n            predicted_labels=np.concatenate(all_predicted_labels).astype(np.int64),\n            boxes=np.concatenate(all_boxes).astype(np.float64),\n            objectness=np.concatenate(all_objectness).astype(np.float64),\n        )\n\n        metadata = {\n            \"checkpoint\": str(Path(args.checkpoint).resolve()),\n            \"dataset\": prob_args.dataset,\n            \"data_root\": prob_args.data_root,\n            \"image_count\": len(image_ids),\n            \"proposal_count\": len(all_image_ids),\n            \"feature_dimension\": int(embeddings.shape[1]),\n            \"previous_introduced_classes\": prob_args.PREV_INTRODUCED_CLS,\n            \"current_introduced_classes\": prob_args.CUR_INTRODUCED_CLS,\n            \"unknown_class_index\": prob_args.num_classes - 1,\n            \"objectness_temperature\": prob_args.obj_temp,\n            \"maximum_proposals_per_image\": args.max_proposals_per_image,\n            \"minimum_unknown_score\": args.minimum_unknown_score,\n        }\n        output_path.with_suffix(\".json\").write_text(\n            json.dumps(metadata, indent=2),\n            encoding=\"utf-8\",\n        )\n\n        print(f\"Saved proposals: {output_path}\")\n        print(f\"Images: {len(image_ids)}\")\n        print(f\"Proposals: {len(all_image_ids)}\")\n        print(\"Feature dimension:\", metadata[\"feature_dimension\"])\n    finally:\n        if not args.keep_temporary_split:\n            remove_temporary_split(split_path)\n\n\ndef check_command(_: argparse.Namespace) -> None:\n    check_feature_patch()\n\n\ndef unsupported_command(args: argparse.Namespace) -> None:\n    raise SystemExit(\n        f\"{args.command!r} is not connected in daowod_prob_bridge.py. Use the official \"\n        \"PROB scripts directly or wire this subcommand to the protocol-specific command.\"\n    )\n\n\ndef build_parser() -> argparse.ArgumentParser:\n    parser = argparse.ArgumentParser(description=\"DAOWOD bridge for the official PROB repository.\")\n    subparsers = parser.add_subparsers(dest=\"command\", required=True)\n\n    check_parser = subparsers.add_parser(\"check\", help=\"Check that the feature patch is present.\")\n    check_parser.set_defaults(function=check_command)\n\n    train_parser = subparsers.add_parser(\"train\", help=\"Unsupported placeholder.\")\n    train_parser.add_argument(\"--labelled-ids\", required=True)\n    train_parser.add_argument(\"--previous-checkpoint\", default=\"\")\n    train_parser.add_argument(\"--output-checkpoint\", required=True)\n    train_parser.add_argument(\"--output-dir\", required=True)\n    train_parser.add_argument(\"--seed\", type=int, default=0)\n    train_parser.set_defaults(function=unsupported_command)\n\n    evaluate_parser = subparsers.add_parser(\"evaluate\", help=\"Unsupported placeholder.\")\n    evaluate_parser.add_argument(\"--checkpoint\", required=True)\n    evaluate_parser.add_argument(\"--output\", required=True)\n    evaluate_parser.set_defaults(function=unsupported_command)\n\n    predict_parser = subparsers.add_parser(\n        \"predict\",\n        help=\"Export proposal scores and decoder embeddings.\",\n    )\n    predict_parser.add_argument(\"--image-ids\", required=True)\n    predict_parser.add_argument(\"--checkpoint\", required=True)\n    predict_parser.add_argument(\"--output\", required=True)\n    predict_parser.add_argument(\"--data-root\", default=\"./data/OWOD\")\n    predict_parser.add_argument(\n        \"--dataset\",\n        choices=(\"TOWOD\", \"OWDETR\", \"VOC2007\"),\n        default=\"TOWOD\",\n    )\n    predict_parser.add_argument(\"--prev-introduced-classes\", type=int, default=0)\n    predict_parser.add_argument(\"--current-introduced-classes\", type=int, default=20)\n    predict_parser.add_argument(\"--num-classes\", type=int, default=81)\n    predict_parser.add_argument(\"--objectness-temperature\", type=float, default=1.3)\n    predict_parser.add_argument(\"--max-proposals-per-image\", type=int, default=20)\n    predict_parser.add_argument(\"--minimum-unknown-score\", type=float, default=0.0)\n    predict_parser.add_argument(\"--batch-size\", type=int, default=1)\n    predict_parser.add_argument(\"--num-workers\", type=int, default=2)\n    predict_parser.add_argument(\"--device\", default=\"cuda\")\n    predict_parser.add_argument(\"--seed\", type=int, default=42)\n    predict_parser.add_argument(\"--keep-temporary-split\", action=\"store_true\")\n    predict_parser.set_defaults(function=export_proposals)\n\n    return parser\n\n\ndef main() -> None:\n    args = build_parser().parse_args()\n    args.function(args)\n\n\nif __name__ == \"__main__\":\n    main()\n"

def write_prob_bridge(prob_path: Path) -> Path:
    bridge_path = prob_path / "daowod_prob_bridge.py"
    bridge_path.write_text(BRIDGE_SOURCE, encoding="utf-8")
    compile(BRIDGE_SOURCE, str(bridge_path), "exec")
    return bridge_path

def patch_prob_model_features(prob_path: Path) -> Path:
    model_path = prob_path / "models" / "prob_deformable_detr.py"
    source = model_path.read_text(encoding="utf-8")
    if "'pred_features': hs[-1]" in source or '"pred_features": hs[-1]' in source:
        compile(source, str(model_path), "exec")
        return model_path

    lines = source.splitlines()
    patched_lines = []
    patched = False
    for line in lines:
        if (
            "out = {'pred_logits': outputs_class[-1]" in line
            and "'pred_obj':outputs_objectness[-1]" in line
        ):
            indent = line[: len(line) - len(line.lstrip())]
            patched_lines.extend(
                [
                    indent + "out = {",
                    indent + "    'pred_logits': outputs_class[-1],",
                    indent + "    'pred_boxes': outputs_coord[-1],",
                    indent + "    'pred_obj': outputs_objectness[-1],",
                    indent + "    'pred_features': hs[-1],",
                    indent + "}",
                ]
            )
            patched = True
        else:
            patched_lines.append(line)
    if not patched:
        raise RuntimeError("Could not locate the PROB output dictionary to patch.")
    patched_source = "\n".join(patched_lines) + "\n"
    compile(patched_source, str(model_path), "exec")
    model_path.write_text(patched_source, encoding="utf-8")
    return model_path

if PROB_PATH.exists():
    write_prob_bridge(PROB_PATH)
    patch_prob_model_features(PROB_PATH)
    STATUS["PROB patch"] = "OK"
    run_command([sys.executable, "daowod_prob_bridge.py", "check"], cwd=PROB_PATH, timeout=120)
    STATUS["bridge check"] = "OK"
else:
    STATUS["PROB patch"] = "MISSING"
    STATUS["bridge check"] = "MISSING"
    print("PROB checkout is unavailable, so the bridge patch was skipped.")

## 11. PROB Attention Compatibility

In [ ]:
ATTENTION_BACKEND_OK = False
ATTENTION_BACKEND_DETAIL = "not checked"

if not PROB_PATH.exists():
    STATUS["PROB attention backend"] = "MISSING"
    ATTENTION_BACKEND_DETAIL = "PROB checkout missing"
else:
    run_command(
        [sys.executable, "-m", "py_compile", "models/ops/functions/ms_deform_attn_func.py"],
        cwd=PROB_PATH,
        timeout=120,
        check=True,
    )
    run_command(
        [sys.executable, "-m", "py_compile", "models/ops/modules/ms_deform_attn.py"],
        cwd=PROB_PATH,
        timeout=120,
        check=True,
    )
    tiny_forward = '''
import torch
from models.ops.modules.ms_deform_attn import MSDeformAttn
module = MSDeformAttn(d_model=8, n_levels=1, n_heads=2, n_points=2)
query = torch.zeros(1, 3, 8)
reference_points = torch.full((1, 3, 1, 2), 0.5)
input_flatten = torch.zeros(1, 4, 8)
input_spatial_shapes = torch.tensor([[2, 2]], dtype=torch.long)
input_level_start_index = torch.tensor([0], dtype=torch.long)
output = module(query, reference_points, input_flatten, input_spatial_shapes, input_level_start_index)
assert tuple(output.shape) == (1, 3, 8)
print("tiny forward shape:", tuple(output.shape))
'''
    result = run_command([sys.executable, "-c", tiny_forward], cwd=PROB_PATH, timeout=120, check=False)
    if result.returncode == 0:
        ATTENTION_BACKEND_OK = True
        ATTENTION_BACKEND_DETAIL = "official extension import and tiny forward passed"
        STATUS["PROB attention backend"] = "OK"
    else:
        function_source = (PROB_PATH / "models/ops/functions/ms_deform_attn_func.py").read_text(encoding="utf-8")
        module_source = (PROB_PATH / "models/ops/modules/ms_deform_attn.py").read_text(encoding="utf-8")
        assert "def ms_deform_attn_core_pytorch(value, value_spatial_shapes, sampling_locations, attention_weights)" in function_source
        assert "def forward(self, query, reference_points, input_flatten, input_spatial_shapes, input_level_start_index, input_padding_mask=None)" in module_source
        ATTENTION_BACKEND_DETAIL = (
            "official CUDA extension is unavailable here; pure-PyTorch fallback exists but was not patched in this notebook"
        )
        STATUS["PROB attention backend"] = "SKIPPED"

print("Attention backend:", ATTENTION_BACKEND_DETAIL)
print("Real PROB inference will run only when this backend check is OK.")

## 12. Google Drive And Data Validation

In [ ]:
if IN_COLAB:
    try:
        from google.colab import drive  # type: ignore[import-not-found]

        drive.mount("/content/drive", force_remount=False)
        STATUS["Drive"] = "OK"
    except Exception as error:
        STATUS["Drive"] = "FAILED"
        print("Drive mount failed:", error)
else:
    STATUS["Drive"] = "SKIPPED"
    print("Local execution: Google Drive mount skipped.")

required_paths = {
    "Drive root": DRIVE_ROOT,
    "JPEGImages": DRIVE_DATA_ROOT / "JPEGImages",
    "Annotations": DRIVE_DATA_ROOT / "Annotations",
    "ImageSets/TOWOD": DRIVE_DATA_ROOT / "ImageSets" / "TOWOD",
    "checkpoint": DRIVE_CHECKPOINT,
}
path_rows = []
for label, path in required_paths.items():
    exists = path.exists()
    path_rows.append({"path": label, "status": "OK" if exists else "MISSING", "location": str(path)})

display(pd.DataFrame(path_rows))
dataset_ready = all(row["status"] == "OK" for row in path_rows if row["path"] != "checkpoint")
checkpoint_ready = DRIVE_CHECKPOINT.exists()
STATUS["dataset"] = "OK" if dataset_ready else "MISSING"
STATUS["checkpoint"] = "OK" if checkpoint_ready else "MISSING"
INFERENCE_READY = bool(dataset_ready and checkpoint_ready and ATTENTION_BACKEND_OK and STATUS["bridge check"] == "OK")
print("INFERENCE_READY =", INFERENCE_READY)

## 13. One-Image Real PROB Export

In [ ]:
REAL_PROPOSAL_PATH = DAOWOD_PATH / "outputs" / "colab_one_image_proposals.npz"

if not INFERENCE_READY:
    STATUS["real proposal export"] = "SKIPPED"
    print("Skipping real proposal export because data, checkpoint, or attention backend is unavailable.")
else:
    split_path = DRIVE_DATA_ROOT / "ImageSets" / TASK_SETTINGS["dataset"] / f"{TASK_SETTINGS['test_split']}.txt"
    image_ids = [line.strip().split()[0] for line in split_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    if not image_ids:
        raise RuntimeError(f"No image IDs found in {split_path}")
    one_image_ids_path = DAOWOD_PATH / "outputs" / "one_image_ids.txt"
    one_image_ids_path.parent.mkdir(parents=True, exist_ok=True)
    one_image_ids_path.write_text(image_ids[0] + "\n", encoding="utf-8")
    run_command(
        [
            sys.executable,
            "daowod_prob_bridge.py",
            "predict",
            "--image-ids",
            one_image_ids_path,
            "--checkpoint",
            DRIVE_CHECKPOINT,
            "--output",
            REAL_PROPOSAL_PATH,
            "--data-root",
            DRIVE_DATA_ROOT,
            "--dataset",
            TASK_SETTINGS["dataset"],
            "--max-proposals-per-image",
            TASK_SETTINGS["max_proposals_per_image"],
            "--minimum-unknown-score",
            TASK_SETTINGS["minimum_unknown_score"],
            "--device",
            "cuda",
        ],
        cwd=PROB_PATH,
        timeout=600,
    )

    with np.load(REAL_PROPOSAL_PATH, allow_pickle=True) as proposal_file:
        required = {"image_ids", "confidence", "embeddings", "posterior", "predicted_labels", "boxes", "objectness"}
        missing = required - set(proposal_file.files)
        if missing:
            raise RuntimeError(f"Missing proposal keys: {sorted(missing)}")
        proposal_count = proposal_file["image_ids"].shape[0]
        assert proposal_count > 0
        for key in required - {"image_ids"}:
            array = proposal_file[key]
            assert array.shape[0] == proposal_count, key
            assert np.all(np.isfinite(array)), key
        assert proposal_file["embeddings"].ndim == 2
        assert set(proposal_file["image_ids"].astype(str)) == {image_ids[0]}
        print("Real proposal count:", proposal_count)
        print("Embedding dimension:", proposal_file["embeddings"].shape[1])
    STATUS["real proposal export"] = "OK"

## 14. Real PROB Feature Scoring

In [ ]:
from daowod.prob_adapter import ProposalBatch

if STATUS["real proposal export"] != "OK" or not REAL_PROPOSAL_PATH.exists():
    STATUS["real feature scoring"] = "SKIPPED"
    print("Skipping real feature scoring because one-image export is unavailable.")
else:
    real_batch = ProposalBatch.load(REAL_PROPOSAL_PATH)
    print(
        "Using proposals from the same image as reference embeddings is only acceptable "
        "for an integration smoke test."
    )
    print(
        "A real acquisition round must export reference embeddings from the currently "
        "labelled image set using the same detector checkpoint."
    )
    real_result = score_proposals(
        strategy="full",
        uncertainty_mode="ambiguity",
        pseudo_label_source="predicted" if real_batch.predicted_labels is not None else "cluster",
        confidence=real_batch.confidence,
        posterior=real_batch.posterior,
        embeddings=real_batch.embeddings,
        reference_embeddings=real_batch.embeddings,
        predicted_labels=real_batch.predicted_labels,
        cluster_count=min(5, real_batch.embeddings.shape[0]),
        neighbour_count=min(5, max(1, real_batch.embeddings.shape[0] - 1)),
        seed=RANDOM_SEED,
        weights=weights,
    )
    real_table = pd.DataFrame(
        {
            "image_id": real_batch.image_ids,
            "uncertainty": real_result.uncertainty,
            "novelty": real_result.novelty,
            "pseudo_label": real_result.pseudo_labels,
            "rarity": real_result.rarity,
            "coherence": real_result.coherence,
            "rarity_bonus": real_result.rarity * real_result.coherence**weights.coherence_power,
            "full_score": real_result.scores,
        }
    )
    assert np.all(np.isfinite(real_result.scores))
    assert len(real_table) == real_batch.embeddings.shape[0]
    display(real_table.head(20).round(4))
    STATUS["real feature scoring"] = "OK"

## 15. Active-Learning Loop Preview

The intended orchestration is:

```text
for each seed:
    build identical controlled pool
    for each strategy:
        initialise identical labelled set
        for each round:
            train
            evaluate
            export candidate proposals
            export labelled reference proposals
            select next image batch
            reveal labels
```

`ActiveLearningExperiment` implements this orchestration boundary, but the PROB bridge intentionally does not fake full train/evaluate commands. Real PROB train/evaluate integration is pending.

In [ ]:
from daowod import ActiveLearningExperiment, ProbAdapter

print("Experiment API available:", ActiveLearningExperiment.__name__, ProbAdapter.__name__)
print("Dry-run only: no large training or fake metrics are launched.")

## 16. Final Status Report

In [ ]:
final_status_order = [
    "GPU",
    "DAOWOD clone",
    "DAOWOD installation",
    "Ruff",
    "pytest",
    "compileall",
    "synthetic scoring",
    "ablation demo",
    "dataset-state demo",
    "grouped metrics demo",
    "PROB clone",
    "PROB patch",
    "bridge check",
    "PROB attention backend",
    "Drive",
    "dataset",
    "checkpoint",
    "real proposal export",
    "real feature scoring",
    "full train/evaluate loop",
]
status_table = pd.DataFrame(
    [{"item": item, "status": STATUS[item]} for item in final_status_order]
)
display(status_table)
print(
    "Exact next research step: wire protocol-specific PROB train/evaluate commands, "
    "then export candidate and labelled-reference proposal features for a real active-learning round."
)